# Pipeline ETL — DataRetail 360 · Datamart Dimensional

Notebook único, ejecutable en **Google Colab** (`Entorno de ejecución → Ejecutar todo`).

Implementa el flujo ETL completo sobre el modelo dimensional definido en `Tablas.md`:
6 dimensiones + 4 fact tables (esquema estrella, single-PK), con generación de datos
sintéticos (Bronze), cuarentena de rechazos, carga a SQLite, idempotencia diaria y
logging propio con nivel + paso + timestamp.

## Índice

| Paso | Sección |
|---|---|
| 0 | Preparación del entorno y parámetros |
| 1 | Logging personalizado |
| 2 | Generación de datos Bronze (todas las tablas) |
| 3 | EXTRACT |
| 4 | TRANSFORM + rechazos |
| 5 | LOAD a SQLite |
| 6 | Idempotencia diaria |
| 7 | Tests de cada paso |
| 8 | Ejecución integral |

## 0. Preparación del entorno

Dependencias mínimas. El notebook funciona sin paquetes externos más allá de `pandas` y `numpy`,
que ya vienen en Colab.

In [15]:
# Dependencias (Colab ya trae pandas/numpy; se verifica por si acaso)
import importlib.util, subprocess, sys
REQUIRED = {"pandas": "pandas", "numpy": "numpy", "pytest": "pytest"}
missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Instalando:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencias listas ✔")

Dependencias listas ✔


In [16]:
import os, sys, json, time, sqlite3, hashlib, random, logging, re, shutil
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd

# ── Parámetros del laboratorio ──────────────────────────────
SEED              = 42
N_VENTAS          = 200_000
N_INVENTARIO      = 12000
N_CAMPANAS        = 150
N_DEVOLUCIONES    = 3000
N_CLIENTES        = 40000
FECHA_INICIO      = "2025-01-01"
FECHA_FIN         = "2025-12-31"
DB_NAME           = "DataRetail360.db"

# Timestamp único de corrida (se usa en CSV, rechazos y logs)
RUN_TS   = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DATE = datetime.now().strftime("%Y-%m-%d")

DATA_DIR     = Path("data")
LOG_DIR      = Path("logs")
REJECTED_DIR = Path("rejected")
for d in (DATA_DIR, LOG_DIR, REJECTED_DIR):
    d.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)
print(f"RUN_TS={RUN_TS} | RUN_DATE={RUN_DATE} | DB={DB_NAME}")

RUN_TS=20261103_021101 | RUN_DATE=2026-11-03 | DB=DataRetail360.db


## 1. Logging personalizado

Requisitos del plan:
- Archivo `.log` con **timestamp** en el nombre.
- Cada línea indica **nivel** (DEBUG/INFO/WARNING/ERROR), **paso** y **timestamp**.
- Los logs se muestran en pantalla mientras corre el notebook.

Códigos de paso usados: `E` = generación/extracción, `T` = transformación,
`L` = carga, `S` = sistema (arranque, tests, orquestación).

In [17]:
class ETLFormatter(logging.Formatter):
    """Formato: timestamp | nivel | paso | mensaje."""
    def format(self, record):
        step = getattr(record, "step", "-")
        return f"{self.formatTime(record, '%Y-%m-%d %H:%M:%S')} | {record.levelname:<8} | [{step}] | {record.getMessage()}"


def setup_logging(log_dir, run_ts):
    """Configura el logger 'etl': consola + archivo .log con timestamp."""
    log = logging.getLogger("etl")
    log.setLevel(logging.DEBUG)
    log.propagate = False
    for h in list(log.handlers):
        log.removeHandler(h)
        h.close()

    fmt = ETLFormatter()
    ch = logging.StreamHandler(sys.stdout)
    ch.setFormatter(fmt); ch.setLevel(logging.DEBUG); log.addHandler(ch)

    log_file = Path(log_dir) / f"etl_{run_ts}.log"
    fh = logging.FileHandler(log_file, encoding="utf-8")
    fh.setFormatter(fmt); fh.setLevel(logging.DEBUG); log.addHandler(fh)
    return log, log_file


log, LOG_FILE = setup_logging(LOG_DIR, RUN_TS)
log.info("Pipeline inicializado", extra={"step": "S"})
log.debug("Parámetros cargados", extra={"step": "S"})
print("Log:", LOG_FILE)

2026-11-03 02:11:01 | INFO     | [S] | Pipeline inicializado
2026-11-03 02:11:01 | DEBUG    | [S] | Parámetros cargados
Log: logs/etl_20261103_021101.log


## 2. Generación de datos Bronze

Se generan **todas las tablas** de `Tablas.md` con código y funciones pseudoaleatorias
(semilla fija). Los RUT se generan con dígito verificador chileno válido en ≥ 90% de los casos.
Cada tabla se persiste como `{tabla}_{timestamp}.csv`.

In [18]:
def dv_rut(n: int) -> str:
    """Calcula el dígito verificador de un RUT chileno (módulo 11)."""
    s, m = 0, 2
    while n:
        s, n, m = s + (n % 10) * m, n // 10, 2 if m == 7 else m + 1
    r = 11 - s % 11
    return "0" if r == 11 else "K" if r == 10 else str(r)


def generar_dim_fecha(inicio: str, fin: str) -> pd.DataFrame:
    """Dimensión fecha completa (día, mes, trimestre, año, fin de semana, feriado)."""
    fechas = pd.date_range(inicio, fin, freq="D")
    return pd.DataFrame({
        "fecha_key":    fechas.strftime("%Y%m%d").astype(int),
        "fecha":        fechas.strftime("%Y-%m-%d"),
        "dia":          fechas.day,
        "mes":          fechas.month,
        "nombre_mes":   fechas.strftime("%B"),
        "trimestre":    (fechas.month - 1) // 3 + 1,
        "anio":         fechas.year,
        "dia_semana":   fechas.dayofweek + 1,
        "nombre_dia":   fechas.strftime("%A"),
        "es_fin_semana": (fechas.dayofweek >= 5).astype(int),
        "es_feriado":   0,
    })


def generar_dim_categoria() -> pd.DataFrame:
    """Categorías de producto."""
    return pd.DataFrame({
        "categoria_key": range(1, 6),
        "categoria": ["Tecnología", "Accesorios", "Gaming", "Electrohogar", "Hogar"],
        "descripcion": [
            "Productos tecnológicos", "Accesorios y periféricos",
            "Consolas y videojuegos", "Electrodomésticos", "Artículos para el hogar",
        ],
    })


def generar_dim_producto(n: int = 20) -> pd.DataFrame:
    """Productos con SKU único y categoría foránea."""
    nombres = [
        "Notebook 14", "Audífonos", "Smart TV 55", "Celular", "Tablet 10",
        "Monitor 27", "Teclado", "Mouse", "Impresora", "Parlante",
        "Smartwatch", "Cámara", "Router WiFi", "Disco SSD 1TB", "Consola",
        "Microondas", "Refrigerador", "Lavadora", "Aspiradora", "Ventilador",
    ]
    skus = [f"P{i+1:03d}" for i in range(n)]
    return pd.DataFrame({
        "producto_key":   range(1, n + 1),
        "sku":            skus,
        "producto":       nombres[:n],
        "categoria_key":  rng.integers(1, 6, n),
        "precio_venta":   rng.integers(15_000, 800_000, n),
        "costo_unitario": rng.integers(8_000, 500_000, n),
    })


def generar_dim_sucursal() -> pd.DataFrame:
    """Sucursales con comuna, región y horario."""
    data = [
        ("Providencia", "Providencia", "Metropolitana", "09:00-21:00"),
        ("Las Condes", "Las Condes", "Metropolitana", "10:00-22:00"),
        ("Maipú", "Maipú", "Metropolitana", "09:00-21:00"),
        ("Ñuñoa", "Ñuñoa", "Metropolitana", "09:00-21:00"),
        ("Concepción", "Concepción", "Biobío", "09:00-21:00"),
        ("Antofagasta", "Antofagasta", "Antofagasta", "09:00-21:00"),
        ("Viña del Mar", "Viña del Mar", "Valparaíso", "10:00-22:00"),
        ("Temuco", "Temuco", "Araucanía", "09:00-21:00"),
    ]
    return pd.DataFrame({
        "sucursal_key":     range(1, len(data) + 1),
        "sucursal":         [d[0] for d in data],
        "comuna":           [d[1] for d in data],
        "region":           [d[2] for d in data],
        "horario_atencion": [d[3] for d in data],
    })


def generar_dim_canal() -> pd.DataFrame:
    """Canales de venta."""
    return pd.DataFrame({
        "id_canal":    range(1, 4),
        "descripcion": ["Tienda física", "E-commerce", "App"],
    })


def generar_dim_cliente(n: int = 500) -> pd.DataFrame:
    """Clientes con RUT chileno válido ≥ 90% (10% con DV alterado a propósito)."""
    rut_nums = rng.choice(np.arange(6_000_000, 26_000_000), n, replace=False)  # únicos
    dv = [dv_rut(int(r)) for r in rut_nums]
    for i in range(n):
        if rng.random() < 0.10:                 # 10% inválido
            dv[i] = "0" if dv[i] != "0" else "1"
    nombres = ["Ana", "Carlos", "María", "Pedro", "Lucía", "Jorge", "Carmen", "Luis", "Sofía", "Diego"]
    apellidos = ["Pérez", "González", "Rodríguez", "López", "Martínez", "García", "Sánchez", "Ramírez", "Torres", "Flores"]
    return pd.DataFrame({
        "cliente_key":  range(1, n + 1),
        "cliente_id":   range(1, n + 1),
        "rut_cliente":  rut_nums,
        "dv_cliente":   dv,
        "nombre":       [f"{nombres[i % 10]} {apellidos[i % 10]}" for i in range(n)],
        "region":       rng.choice(["Metropolitana", "Biobío", "Antofagasta", "Valparaíso", "Araucanía"], n),
        "valido_desde": "2025-01-01",
        "valido_hasta": None,
        "es_actual":    1,
    })

In [19]:
def generar_fact_ventas(n, dim_fecha, dim_producto, dim_sucursal, dim_cliente, dim_canal):
    """Hecho ventas: todas las 5 dimensiones (X en Bus Matrix)."""
    fk_fecha = rng.choice(dim_fecha["fecha_key"].values, n)
    fk_prod  = rng.choice(dim_producto["producto_key"].values, n)
    fk_suc   = rng.choice(dim_sucursal["sucursal_key"].values, n)
    fk_cli   = rng.choice(dim_cliente["cliente_key"].values, n)
    fk_can   = rng.choice(dim_canal["id_canal"].values, n)
    cantidad = rng.integers(1, 4, n)
    precios  = dim_producto.set_index("producto_key").loc[fk_prod, "precio_venta"].values
    bruto    = precios * cantidad
    desc     = np.where(rng.random(n) < 0.20, np.round(bruto * 0.10, 2), 0.0)
    estado   = rng.choice(["OK", "DEVUELTA", "ANULADA"], n, p=[0.90, 0.08, 0.02])
    return pd.DataFrame({
        "fact_key":        range(1, n + 1),
        "fecha_key":       fk_fecha,
        "producto_key":    fk_prod,
        "sucursal_key":    fk_suc,
        "cliente_key":     fk_cli,
        "id_canal":        fk_can,
        "cantidad":        cantidad,
        "monto_bruto":     np.round(bruto, 2),
        "monto_neto":      np.round(np.where(estado == "OK", bruto - desc, 0.0), 2),
        "estado":          estado,
        "monto_descuento": desc,
    })


def generar_fact_inventario(n, dim_fecha, dim_producto, dim_sucursal):
    """Hecho inventario: Fecha, Producto, Sucursal (Canal O, Cliente O → sin columnas)."""
    return pd.DataFrame({
        "fact_key":            range(1, n + 1),
        "fecha_key":           rng.choice(dim_fecha["fecha_key"].values, n),
        "producto_key":        rng.choice(dim_producto["producto_key"].values, n),
        "sucursal_key":        rng.choice(dim_sucursal["sucursal_key"].values, n),
        "cantidad_stock":      rng.integers(0, 500, n),
        "stock_reposicion":    rng.integers(0, 200, n),
        "ultima_actualizacion": rng.choice(dim_fecha["fecha"].values, n),
    })


def generar_fact_campanas(n, dim_fecha, dim_producto, dim_cliente, dim_canal):
    """Hecho campañas: Fecha, Producto, Canal, Cliente (Sucursal O → sin columna)."""
    inicio = rng.choice(dim_fecha["fecha"].values, n)
    return pd.DataFrame({
        "fact_key":              range(1, n + 1),
        "fecha_key":             rng.choice(dim_fecha["fecha_key"].values, n),
        "producto_key":          rng.choice(dim_producto["producto_key"].values, n),
        "id_canal":              rng.choice(dim_canal["id_canal"].values, n),
        "cliente_key":           rng.choice(dim_cliente["cliente_key"].values, n),
        "monto_inversion":       rng.integers(100_000, 5_000_000, n),
        "numero_interacciones":  rng.integers(10, 10_000, n),
        "fecha_inicio_campaña":  inicio,
        "fecha_termino_campaña": (pd.to_datetime(inicio) + pd.to_timedelta(rng.integers(7, 90, n), unit="D")).strftime("%Y-%m-%d"),
    })


def generar_fact_devoluciones(n, dim_fecha, dim_producto, dim_sucursal, dim_cliente, dim_canal):
    """Hecho devoluciones: todas las 5 dimensiones (X en Bus Matrix)."""
    return pd.DataFrame({
        "fact_key":      range(1, n + 1),
        "fecha_key":     rng.choice(dim_fecha["fecha_key"].values, n),
        "producto_key":  rng.choice(dim_producto["producto_key"].values, n),
        "sucursal_key":  rng.choice(dim_sucursal["sucursal_key"].values, n),
        "cliente_key":   rng.choice(dim_cliente["cliente_key"].values, n),
        "id_canal":      rng.choice(dim_canal["id_canal"].values, n),
        "id_devolucion": range(1, n + 1),
        "motivo":        rng.choice(["Producto defectuoso", "Talla incorrecta", "No era lo esperado", "Llegó dañado"], n),
        "monto":         rng.integers(10_000, 500_000, n),
        "estado":        rng.choice(["PROCESADA", "PENDIENTE"], n, p=[0.70, 0.30]),
    })

In [20]:
def guardar_csv(df: pd.DataFrame, nombre: str, ts: str) -> Path:
    """Persiste una tabla como CSV con timestamp en el nombre."""
    path = DATA_DIR / f"{nombre}_{ts}.csv"
    df.to_csv(path, index=False)
    log.info("Bronze %s → %s (%d filas)", nombre, path.name, len(df), extra={"step": "E"})
    return path


def generar_bronze(ts: str = RUN_TS) -> dict:
    """Genera TODAS las tablas del datamart y las persiste como CSV con timestamp."""
    log.info("Generando datos Bronze (todas las tablas)", extra={"step": "E"})
    dim_fecha     = generar_dim_fecha(FECHA_INICIO, FECHA_FIN)
    dim_categoria = generar_dim_categoria()
    dim_producto  = generar_dim_producto()
    dim_sucursal  = generar_dim_sucursal()
    dim_canal     = generar_dim_canal()
    dim_cliente   = generar_dim_cliente(N_CLIENTES)

    tablas = {
        "dim_fecha":         dim_fecha,
        "dim_categoria":     dim_categoria,
        "dim_producto":      dim_producto,
        "dim_sucursal":      dim_sucursal,
        "dim_cliente":       dim_cliente,
        "dim_canal":         dim_canal,
        "fact_ventas":       generar_fact_ventas(N_VENTAS, dim_fecha, dim_producto, dim_sucursal, dim_cliente, dim_canal),
        "fact_inventario":   generar_fact_inventario(N_INVENTARIO, dim_fecha, dim_producto, dim_sucursal),
        "fact_campanas":     generar_fact_campanas(N_CAMPANAS, dim_fecha, dim_producto, dim_cliente, dim_canal),
        "fact_devoluciones": generar_fact_devoluciones(N_DEVOLUCIONES, dim_fecha, dim_producto, dim_sucursal, dim_cliente, dim_canal),
    }
    for nombre, df in tablas.items():
        guardar_csv(df, nombre, ts)
    log.info("Bronze completo: %d tablas", len(tablas), extra={"step": "E"})
    return tablas

BRONZE = generar_bronze()

2026-11-03 02:11:01 | INFO     | [E] | Generando datos Bronze (todas las tablas)
2026-11-03 02:11:01 | INFO     | [E] | Bronze dim_fecha → dim_fecha_20261103_021101.csv (365 filas)
2026-11-03 02:11:01 | INFO     | [E] | Bronze dim_categoria → dim_categoria_20261103_021101.csv (5 filas)
2026-11-03 02:11:01 | INFO     | [E] | Bronze dim_producto → dim_producto_20261103_021101.csv (20 filas)
2026-11-03 02:11:01 | INFO     | [E] | Bronze dim_sucursal → dim_sucursal_20261103_021101.csv (8 filas)
2026-11-03 02:11:01 | INFO     | [E] | Bronze dim_cliente → dim_cliente_20261103_021101.csv (40000 filas)
2026-11-03 02:11:01 | INFO     | [E] | Bronze dim_canal → dim_canal_20261103_021101.csv (3 filas)


/tmp/ipykernel_185231/1493486422.py:21: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  "dia_semana":   fechas.dayofweek + 1,
/tmp/ipykernel_185231/1493486422.py:23: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  "es_fin_semana": (fechas.dayofweek >= 5).astype(int),


2026-11-03 02:11:01 | INFO     | [E] | Bronze fact_ventas → fact_ventas_20261103_021101.csv (200000 filas)
2026-11-03 02:11:01 | INFO     | [E] | Bronze fact_inventario → fact_inventario_20261103_021101.csv (12000 filas)
2026-11-03 02:11:01 | INFO     | [E] | Bronze fact_campanas → fact_campanas_20261103_021101.csv (150 filas)
2026-11-03 02:11:01 | INFO     | [E] | Bronze fact_devoluciones → fact_devoluciones_20261103_021101.csv (3000 filas)
2026-11-03 02:11:01 | INFO     | [E] | Bronze completo: 10 tablas


In [21]:
# Vista rápida de las tablas generadas
resumen = pd.DataFrame({
    "tabla": list(BRONZE.keys()),
    "filas": [len(df) for df in BRONZE.values()],
    "columnas": [len(df.columns) for df in BRONZE.values()],
})
display(resumen)

# Verificación RUT válido ≥ 90%
clientes = BRONZE["dim_cliente"]
validos = sum(dv_rut(int(r)) == d for r, d in zip(clientes["rut_cliente"], clientes["dv_cliente"]))
print(f"RUT válidos: {validos}/{len(clientes)} = {validos/len(clientes):.1%}")
display(BRONZE["fact_inventario"].head(3))
display(BRONZE["fact_campanas"].head(3))

,tabla,filas,columnas
0,dim_fecha,365,11
1,dim_categoria,5,3
2,dim_producto,20,6
3,dim_sucursal,8,5
4,dim_cliente,40000,9
5,dim_canal,3,2
6,fact_ventas,200000,11
7,fact_inventario,12000,7
8,fact_campanas,150,9
9,fact_devoluciones,3000,10


RUT válidos: 36035/40000 = 90.1%


,fact_key,fecha_key,producto_key,sucursal_key,cantidad_stock,stock_reposicion,ultima_actualizacion
0,1,20250827,7,4,174,62,2025-08-08
1,2,20251216,20,2,450,162,2025-10-31
2,3,20251031,18,3,87,184,2025-05-10


,fact_key,fecha_key,producto_key,id_canal,cliente_key,monto_inversion,numero_interacciones,fecha_inicio_campaña,fecha_termino_campaña
0,1,20250902,2,2,10868,1580147,74,2025-11-26,2026-02-04
1,2,20251222,8,1,33018,1712428,41,2025-01-03,2025-02-25
2,3,20250609,17,1,2549,4400310,5652,2025-02-11,2025-05-04


## 3. EXTRACT

Lee los CSV con timestamp. Si un archivo no existe (p. ej. ejecución en frío), lo regenera
y lo persiste antes de leerlo, cumpliendo el punto 3.2 del plan.

In [22]:
TABLAS = ["dim_fecha", "dim_categoria", "dim_producto", "dim_sucursal",
          "dim_cliente", "dim_canal", "fact_ventas", "fact_inventario",
          "fact_campanas", "fact_devoluciones"]


def extraer(ts: str, tablas=TABLAS) -> dict:
    """EXTRACT: lee cada CSV con timestamp; si falta, genera y persiste la tabla."""
    log.info("EXTRACT inicio (%d tablas)", len(tablas), extra={"step": "E"})
    datos = {}
    for nombre in tablas:
        path = DATA_DIR / f"{nombre}_{ts}.csv"
        if not path.exists():
            log.warning("%s no existe; se regenera Bronze y se persiste", nombre, extra={"step": "E"})
            generar_bronze(ts)
        df = pd.read_csv(path)
        datos[nombre] = df
        log.info("EXTRACT %-18s %5d filas", nombre, len(df), extra={"step": "E"})
    log.info("EXTRACT fin", extra={"step": "E"})
    return datos

EXTRACTED = extraer(RUN_TS)

2026-11-03 02:11:02 | INFO     | [E] | EXTRACT inicio (10 tablas)
2026-11-03 02:11:02 | INFO     | [E] | EXTRACT dim_fecha            365 filas
2026-11-03 02:11:02 | INFO     | [E] | EXTRACT dim_categoria          5 filas
2026-11-03 02:11:02 | INFO     | [E] | EXTRACT dim_producto          20 filas
2026-11-03 02:11:02 | INFO     | [E] | EXTRACT dim_sucursal           8 filas
2026-11-03 02:11:02 | INFO     | [E] | EXTRACT dim_cliente        40000 filas
2026-11-03 02:11:02 | INFO     | [E] | EXTRACT dim_canal              3 filas
2026-11-03 02:11:02 | INFO     | [E] | EXTRACT fact_ventas        200000 filas
2026-11-03 02:11:02 | INFO     | [E] | EXTRACT fact_inventario    12000 filas
2026-11-03 02:11:02 | INFO     | [E] | EXTRACT fact_campanas        150 filas
2026-11-03 02:11:02 | INFO     | [E] | EXTRACT fact_devoluciones   3000 filas
2026-11-03 02:11:02 | INFO     | [E] | EXTRACT fin


## 4. TRANSFORM + cuarentena de rechazos

Reglas de calidad mapeadas en código. Cada fila rechazada conserva la columna
`motivo_rechazo` y se persiste en `rejected/Rejected_{timestamp}.csv`.

Reglas aplicadas:
- **Claves nulas** en dimensiones → rechazo.
- **FK nulas** en hechos → rechazo.
- **RUT inválido** en `dim_cliente` → rechazo.
- **Montos negativos** o `monto_neto > monto_bruto` en `fact_ventas` → rechazo.
- **Cantidades no positivas** en hechos → rechazo.
- **Estado desconocido** → rechazo.
- **Duplicados** por clave natural (`cliente_id`, `sku`, `fact_key`) → rechazo (se conserva el primero).

In [23]:
REGLAS = {
    "clave_nula":          "Clave primaria o natural nula",
    "fk_nula":             "Clave foránea nula en tabla de hechos",
    "rut_invalido":        "RUT chileno con dígito verificador inválido",
    "monto_negativo":      "Monto negativo",
    "monto_neto_invalido": "monto_neto mayor que monto_bruto o negativo",
    "cantidad_invalida":   "Cantidad menor o igual a cero",
    "estado_desconocido":  "Estado fuera del dominio permitido",
    "duplicado":           "Clave natural duplicada (se conserva la primera)",
}

display(pd.DataFrame(
    {"motivo_rechazo": list(REGLAS.keys()), "descripcion": list(REGLAS.values())}
))


def validar_dimensiones(datos: dict) -> tuple:
    """Valida dimensiones; devuelve (dimensiones_ok, rechazos)."""
    log.info("TRANSFORM dimensiones", extra={"step": "T"})
    ok, rechazos = {}, []
    for nombre, df in datos.items():
        if not nombre.startswith("dim_"):
            continue
        df = df.copy()
        malas = []
        # Clave primaria nula (la primera columna es siempre la surrogate key)
        pk = df.columns[0]
        m = df[pk].isna()
        if m.any():
            malas.append(df[m].assign(motivo_rechazo="clave_nula"))
        # RUT inválido
        if nombre == "dim_cliente":
            invalidos = ~np.array([dv_rut(int(r)) == d for r, d in zip(df["rut_cliente"], df["dv_cliente"])])
            if invalidos.any():
                malas.append(df[invalidos].assign(motivo_rechazo="rut_invalido"))
        # Duplicado por clave natural
        nat = "sku" if nombre == "dim_producto" else ("cliente_id" if nombre == "dim_cliente" else None)
        if nat:
            dup = df.duplicated(subset=[nat], keep="first")
            if dup.any():
                malas.append(df[dup].assign(motivo_rechazo="duplicado"))
        if malas:
            r = pd.concat(malas)
            rechazos.append(r)
            df = df.drop(index=r.index.unique())
        ok[nombre] = df
        log.info("TRANSFORM %-16s %5d ok", nombre, len(df), extra={"step": "T"})
    return ok, rechazos


def validar_hechos(datos: dict, dims_ok: dict) -> tuple:
    """Valida hechos contra claves de dimensión y reglas de negocio."""
    log.info("TRANSFORM hechos", extra={"step": "T"})
    ok, rechazos = {}, []
    fk_map = {
        "fecha_key": "dim_fecha", "producto_key": "dim_producto",
        "sucursal_key": "dim_sucursal", "cliente_key": "dim_cliente",
        "id_canal": "dim_canal",
    }
    for nombre, df in datos.items():
        if not nombre.startswith("fact_"):
            continue
        df = df.copy()
        malas = []

        # FK nula o inexistente en dimensión
        for fk, dim in fk_map.items():
            if fk in df.columns:
                validos = set(dims_ok[dim].iloc[:, 0].astype(int))
                m = df[fk].isna() | ~df[fk].astype("Int64").isin(validos)
                if m.any():
                    malas.append(df[m].assign(motivo_rechazo="fk_nula"))

        # Cantidad no positiva
        for col in ("cantidad", "cantidad_stock", "numero_interacciones"):
            if col in df.columns:
                m = ~(pd.to_numeric(df[col], errors="coerce") > 0)
                if m.any():
                    malas.append(df[m].assign(motivo_rechazo="cantidad_invalida"))

        # Montos
        if "monto_bruto" in df.columns:
            m = (pd.to_numeric(df["monto_bruto"], errors="coerce") < 0) | \
                (pd.to_numeric(df["monto_neto"], errors="coerce") < 0) | \
                (pd.to_numeric(df["monto_neto"], errors="coerce") > pd.to_numeric(df["monto_bruto"], errors="coerce"))
            if m.any():
                malas.append(df[m].assign(motivo_rechazo="monto_negativo"))
        if "monto" in df.columns:
            m = pd.to_numeric(df["monto"], errors="coerce") < 0
            if m.any():
                malas.append(df[m].assign(motivo_rechazo="monto_negativo"))

        # Estado
        if "estado" in df.columns:
            permitidos = {"OK", "DEVUELTA", "ANULADA", "PROCESADA", "PENDIENTE"}
            m = ~df["estado"].isin(permitidos)
            if m.any():
                malas.append(df[m].assign(motivo_rechazo="estado_desconocido"))

        # Duplicado por fact_key
        dup = df.duplicated(subset=["fact_key"], keep="first")
        if dup.any():
            malas.append(df[dup].assign(motivo_rechazo="duplicado"))

        if malas:
            r = pd.concat(malas)
            rechazos.append(r)
            df = df.drop(index=r.index.unique())
        ok[nombre] = df
        log.info("TRANSFORM %-18s %5d ok", nombre, len(df), extra={"step": "T"})
    return ok, rechazos


def guardar_rechazos(rechazos: list, ts: str) -> Path | None:
    """Persiste rechazos con timestamp y motivo_rechazo."""
    if not rechazos:
        log.info("Sin rechazos", extra={"step": "T"})
        return None
    rech = pd.concat(rechazos, ignore_index=True)
    path = REJECTED_DIR / f"Rejected_{ts}.csv"
    rech.to_csv(path, index=False)
    log.warning("Rechazos: %d filas → %s", len(rech), path.name, extra={"step": "T"})
    return path

DIMS_OK, RECHAZOS_DIM = validar_dimensiones(EXTRACTED)
FACTS_OK, RECHAZOS_FACT = validar_hechos(EXTRACTED, DIMS_OK)
REJECTED_PATH = guardar_rechazos(RECHAZOS_DIM + RECHAZOS_FACT, RUN_TS)

print("\nMotivos de rechazo:")
if REJECTED_PATH:
    display(pd.read_csv(REJECTED_PATH)["motivo_rechazo"].value_counts().to_frame())
else:
    print("(ninguno)")

,motivo_rechazo,descripcion
0,clave_nula,Clave primaria o natural nula
1,fk_nula,Clave foránea nula en tabla de hechos
2,rut_invalido,RUT chileno con dígito verificador inválido
3,monto_negativo,Monto negativo
4,monto_neto_invalido,monto_neto mayor que monto_bruto o negativo
5,cantidad_invalida,Cantidad menor o igual a cero
6,estado_desconocido,Estado fuera del dominio permitido
7,duplicado,Clave natural duplicada (se conserva la primera)


2026-11-03 02:11:02 | INFO     | [T] | TRANSFORM dimensiones
2026-11-03 02:11:02 | INFO     | [T] | TRANSFORM dim_fecha          365 ok
2026-11-03 02:11:02 | INFO     | [T] | TRANSFORM dim_categoria        5 ok
2026-11-03 02:11:02 | INFO     | [T] | TRANSFORM dim_producto        20 ok
2026-11-03 02:11:02 | INFO     | [T] | TRANSFORM dim_sucursal         8 ok
2026-11-03 02:11:02 | INFO     | [T] | TRANSFORM dim_cliente      36035 ok
2026-11-03 02:11:02 | INFO     | [T] | TRANSFORM dim_canal            3 ok
2026-11-03 02:11:02 | INFO     | [T] | TRANSFORM hechos
2026-11-03 02:11:02 | INFO     | [T] | TRANSFORM fact_ventas        180072 ok
2026-11-03 02:11:02 | INFO     | [T] | TRANSFORM fact_inventario    11985 ok
2026-11-03 02:11:02 | INFO     | [T] | TRANSFORM fact_campanas        130 ok
2026-11-03 02:11:02 | INFO     | [T] | TRANSFORM fact_devoluciones   2701 ok
2026-11-03 02:11:02 | WARNING  | [T] | Rechazos: 24227 filas → Rejected_20261103_021101.csv

Motivos de rechazo:


,count
motivo_rechazo,
fk_nula,20247
rut_invalido,3965
cantidad_invalida,15


## 5. LOAD a SQLite

Crea/actualiza `DataRetail360.db` con el DDL exacto de `Tablas.md`.
Se cargan **dimensiones primero** y luego hechos, respetando las claves foráneas.

In [24]:
DDL = """
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS dim_fecha (
    fecha_key INTEGER PRIMARY KEY, fecha TEXT NOT NULL UNIQUE,
    dia INTEGER NOT NULL, mes INTEGER NOT NULL, nombre_mes TEXT NOT NULL,
    trimestre INTEGER NOT NULL, anio INTEGER NOT NULL, dia_semana INTEGER NOT NULL,
    nombre_dia TEXT NOT NULL, es_fin_semana INTEGER NOT NULL, es_feriado INTEGER NOT NULL
);
CREATE TABLE IF NOT EXISTS dim_categoria (
    categoria_key INTEGER PRIMARY KEY, categoria TEXT NOT NULL, descripcion TEXT
);
CREATE TABLE IF NOT EXISTS dim_producto (
    producto_key INTEGER PRIMARY KEY, sku TEXT NOT NULL UNIQUE, producto TEXT NOT NULL,
    categoria_key INTEGER NOT NULL REFERENCES dim_categoria(categoria_key),
    precio_venta REAL NOT NULL, costo_unitario REAL NOT NULL
);
CREATE TABLE IF NOT EXISTS dim_sucursal (
    sucursal_key INTEGER PRIMARY KEY, sucursal TEXT NOT NULL, comuna TEXT NOT NULL,
    region TEXT NOT NULL, horario_atencion TEXT
);
CREATE TABLE IF NOT EXISTS dim_cliente (
    cliente_key INTEGER PRIMARY KEY, cliente_id INTEGER NOT NULL UNIQUE,
    rut_cliente INTEGER NOT NULL UNIQUE, dv_cliente TEXT NOT NULL, nombre TEXT NOT NULL,
    region TEXT NOT NULL, valido_desde TEXT NOT NULL, valido_hasta TEXT,
    es_actual INTEGER NOT NULL DEFAULT 1
);
CREATE TABLE IF NOT EXISTS dim_canal (
    id_canal INTEGER PRIMARY KEY, descripcion TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS fact_ventas (
    fact_key INTEGER PRIMARY KEY,
    fecha_key INTEGER NOT NULL REFERENCES dim_fecha(fecha_key),
    producto_key INTEGER NOT NULL REFERENCES dim_producto(producto_key),
    sucursal_key INTEGER NOT NULL REFERENCES dim_sucursal(sucursal_key),
    cliente_key INTEGER NOT NULL REFERENCES dim_cliente(cliente_key),
    id_canal INTEGER NOT NULL REFERENCES dim_canal(id_canal),
    cantidad INTEGER NOT NULL, monto_bruto REAL NOT NULL, monto_neto REAL NOT NULL,
    estado TEXT NOT NULL, monto_descuento REAL DEFAULT 0
);
CREATE TABLE IF NOT EXISTS fact_inventario (
    fact_key INTEGER PRIMARY KEY,
    fecha_key INTEGER NOT NULL REFERENCES dim_fecha(fecha_key),
    producto_key INTEGER NOT NULL REFERENCES dim_producto(producto_key),
    sucursal_key INTEGER NOT NULL REFERENCES dim_sucursal(sucursal_key),
    cantidad_stock INTEGER NOT NULL, stock_reposicion INTEGER,
    ultima_actualizacion TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS fact_campanas (
    fact_key INTEGER PRIMARY KEY,
    fecha_key INTEGER NOT NULL REFERENCES dim_fecha(fecha_key),
    producto_key INTEGER NOT NULL REFERENCES dim_producto(producto_key),
    id_canal INTEGER NOT NULL REFERENCES dim_canal(id_canal),
    cliente_key INTEGER NOT NULL REFERENCES dim_cliente(cliente_key),
    monto_inversion REAL NOT NULL, numero_interacciones INTEGER,
    "fecha_inicio_campaña" TEXT NOT NULL, "fecha_termino_campaña" TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS fact_devoluciones (
    fact_key INTEGER PRIMARY KEY,
    fecha_key INTEGER NOT NULL REFERENCES dim_fecha(fecha_key),
    producto_key INTEGER NOT NULL REFERENCES dim_producto(producto_key),
    sucursal_key INTEGER NOT NULL REFERENCES dim_sucursal(sucursal_key),
    cliente_key INTEGER NOT NULL REFERENCES dim_cliente(cliente_key),
    id_canal INTEGER NOT NULL REFERENCES dim_canal(id_canal),
    id_devolucion INTEGER NOT NULL, motivo TEXT, monto REAL NOT NULL, estado TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS etl_control (
    run_date TEXT PRIMARY KEY, run_id TEXT NOT NULL, ejecutado_en TEXT NOT NULL,
    filas_leidas INTEGER, filas_ok INTEGER, filas_rechazadas INTEGER
);
"""

ORDEN_DIMS  = ["dim_fecha", "dim_categoria", "dim_producto", "dim_sucursal", "dim_cliente", "dim_canal"]
ORDEN_FACTS = ["fact_ventas", "fact_inventario", "fact_campanas", "fact_devoluciones"]


def conectar(db_path: str = DB_NAME) -> sqlite3.Connection:
    """Abre SQLite, activa FKs y aplica el DDL del datamart."""
    con = sqlite3.connect(db_path)
    con.execute("PRAGMA foreign_keys = ON")
    con.executescript(DDL)
    return con


def cargar(dims: dict, facts: dict, db_path: str = DB_NAME) -> dict:
    """LOAD: dimensiones primero, luego hechos, en una transacción."""
    log.info("LOAD inicio", extra={"step": "L"})
    con = conectar(db_path)
    conteos = {}
    try:
        with con:
            for nombre in ORDEN_DIMS:
                if nombre in dims:
                    dims[nombre].to_sql(nombre, con, if_exists="replace", index=False)
                    conteos[nombre] = len(dims[nombre])
                    log.info("LOAD %-16s %5d filas", nombre, len(dims[nombre]), extra={"step": "L"})
            for nombre in ORDEN_FACTS:
                if nombre in facts:
                    facts[nombre].to_sql(nombre, con, if_exists="replace", index=False)
                    conteos[nombre] = len(facts[nombre])
                    log.info("LOAD %-18s %5d filas", nombre, len(facts[nombre]), extra={"step": "L"})
        log.info("LOAD fin", extra={"step": "L"})
    finally:
        con.close()
    return conteos

CONTEOS = cargar(DIMS_OK, FACTS_OK)
print(pd.DataFrame([CONTEOS]).T.rename(columns={0: "filas"}))

2026-11-03 02:11:02 | INFO     | [L] | LOAD inicio
2026-11-03 02:11:02 | INFO     | [L] | LOAD dim_fecha          365 filas
2026-11-03 02:11:02 | INFO     | [L] | LOAD dim_categoria        5 filas
2026-11-03 02:11:02 | INFO     | [L] | LOAD dim_producto        20 filas
2026-11-03 02:11:02 | INFO     | [L] | LOAD dim_sucursal         8 filas
2026-11-03 02:11:02 | INFO     | [L] | LOAD dim_cliente      36035 filas
2026-11-03 02:11:02 | INFO     | [L] | LOAD dim_canal            3 filas
2026-11-03 02:11:02 | INFO     | [L] | LOAD fact_ventas        180072 filas
2026-11-03 02:11:02 | INFO     | [L] | LOAD fact_inventario    11985 filas
2026-11-03 02:11:02 | INFO     | [L] | LOAD fact_campanas        130 filas
2026-11-03 02:11:02 | INFO     | [L] | LOAD fact_devoluciones   2701 filas
2026-11-03 02:11:02 | INFO     | [L] | LOAD fin
                    filas
dim_fecha             365
dim_categoria           5
dim_producto           20
dim_sucursal            8
dim_cliente         36035
dim_ca

## 6. Idempotencia diaria

El pipeline corre **una sola vez por día**. Si ya existe una fila en `etl_control`
para la fecha actual, la ejecución se omite y no modifica datos.
La carga usa `if_exists="replace"`, por lo que incluso un reintento manual produce
el mismo estado final (sin duplicados).

In [25]:
def ya_ejecutado_hoy(db_path: str = DB_NAME, run_date: str = RUN_DATE) -> bool:
    """True si el pipeline ya se ejecutó en la fecha indicada."""
    con = conectar(db_path)
    try:
        return con.execute("SELECT 1 FROM etl_control WHERE run_date = ?", (run_date,)).fetchone() is not None
    finally:
        con.close()


def registrar_ejecucion(run_id: str, filas_leidas: int, filas_ok: int, filas_rechazadas: int,
                        db_path: str = DB_NAME) -> None:
    """Registra la corrida del día para bloquear re-ejecuciones."""
    con = conectar(db_path)
    try:
        con.execute("INSERT OR REPLACE INTO etl_control VALUES (?, ?, ?, ?, ?, ?)",
                    (RUN_DATE, run_id, datetime.now(timezone.utc).isoformat(timespec="seconds"),
                     filas_leidas, filas_ok, filas_rechazadas))
        con.commit()
    finally:
        con.close()

print("¿Ya ejecutado hoy?", ya_ejecutado_hoy())

¿Ya ejecutado hoy? True


## 7. Tests de cada paso

Pruebas autocontenidas (sin archivos externos) que verifican generación, extracción,
transformación, rechazos, carga e idempotencia. Se ejecutan con `run_tests()`.

In [26]:
def _assert(cond, msg):
    """Assert mínimo y autocontenido: falla con `msg` si `cond` es falso."""
    if not cond:
        raise AssertionError(msg)


def test_generacion_bronze():
    """Cada tabla tiene filas y las columnas clave esperadas."""
    _assert(len(BRONZE) == 10, "Deben generarse 10 tablas")
    _assert(len(BRONZE["dim_fecha"]) == 365, "dim_fecha debe cubrir 365 días")
    _assert({"fact_key", "fecha_key", "producto_key"} <= set(BRONZE["fact_ventas"].columns), "Faltan columnas en fact_ventas")
    _assert("id_canal" not in BRONZE["fact_inventario"].columns, "fact_inventario no debe tener id_canal (O)")
    _assert("cliente_key" not in BRONZE["fact_inventario"].columns, "fact_inventario no debe tener cliente_key (O)")
    _assert("sucursal_key" not in BRONZE["fact_campanas"].columns, "fact_campanas no debe tener sucursal_key (O)")


def test_rut_valido_90():
    """Al menos 90% de los RUT generados son válidos."""
    df = BRONZE["dim_cliente"]
    validos = sum(dv_rut(int(r)) == d for r, d in zip(df["rut_cliente"], df["dv_cliente"]))
    _assert(validos / len(df) >= 0.90, f"Solo {validos/len(df):.1%} RUT válidos")


def test_csv_con_timestamp():
    """Los CSV existen y llevan el timestamp de corrida."""
    for nombre in TABLAS:
        _assert((DATA_DIR / f"{nombre}_{RUN_TS}.csv").exists(), f"Falta CSV de {nombre}")


def test_extract_lee_csv():
    """EXTRACT devuelve todas las tablas desde CSV."""
    _assert(set(EXTRACTED) == set(TABLAS), "EXTRACT no devolvió todas las tablas")
    _assert(len(EXTRACTED["fact_ventas"]) == len(BRONZE["fact_ventas"]), "Conteo de ventas distinto al Bronze")


def test_extract_regenera_si_falta():
    """Si falta el CSV, EXTRACT lo regenera, lo persiste y lo extrae (plan 3.2)."""
    import tempfile
    global DATA_DIR
    original = DATA_DIR
    tmp = Path(tempfile.mkdtemp())
    try:
        DATA_DIR = tmp
        datos = extraer("20990101_000000", tablas=["dim_canal"])
        _assert((tmp / "dim_canal_20990101_000000.csv").exists(), "No se regeneró el CSV faltante")
        _assert(len(datos["dim_canal"]) == 3, "dim_canal regenerado con filas incorrectas")
    finally:
        DATA_DIR = original
        shutil.rmtree(tmp, ignore_errors=True)


def test_transform_rechaza_rut_invalido():
    """Los RUT inválidos van a rechazos y no a dim_cliente."""
    n_orig = len(BRONZE["dim_cliente"])
    n_ok = len(DIMS_OK["dim_cliente"])
    _assert(n_ok <= n_orig, "dim_cliente no puede crecer")
    _assert(any("rut_invalido" in r.get("motivo_rechazo", pd.Series()).values for r in RECHAZOS_DIM) or n_ok == n_orig,
            "No se detectaron RUT inválidos")


def test_rechazos_con_motivo():
    """El archivo de rechazos tiene columna motivo_rechazo."""
    if REJECTED_PATH:
        rech = pd.read_csv(REJECTED_PATH)
        _assert("motivo_rechazo" in rech.columns, "Falta motivo_rechazo en Rejected.csv")
        _assert(rech["motivo_rechazo"].notna().all(), "Hay rechazos sin motivo")


def test_motivos_mapeados():
    """Todo motivo_rechazo emitido existe en el diccionario REGLAS del código."""
    if REJECTED_PATH:
        rech = pd.read_csv(REJECTED_PATH)
        usados = set(rech["motivo_rechazo"].dropna().unique())
        _assert(usados <= set(REGLAS), f"Motivos sin mapear: {usados - set(REGLAS)}")
        _assert(len(usados) > 0, "No se registró ningún motivo de rechazo")


def test_load_sqlite():
    """Las 10 tablas existen en SQLite con los conteos cargados."""
    con = conectar()
    try:
        for nombre in ORDEN_DIMS + ORDEN_FACTS:
            n = con.execute(f"SELECT COUNT(*) FROM {nombre}").fetchone()[0]
            _assert(n == CONTEOS[nombre], f"Conteo distinto en {nombre}: {n} != {CONTEOS[nombre]}")
        fk = con.execute("PRAGMA foreign_key_check").fetchall()
        _assert(len(fk) == 0, f"Violaciones de FK: {fk[:3]}")
    finally:
        con.close()


def test_idempotencia_diaria():
    """Registrar dos veces la misma fecha no duplica filas en etl_control (DB temporal)."""
    import tempfile
    tmp = Path(tempfile.mkdtemp()) / "idem.db"
    registrar_ejecucion("test-1", 10, 9, 1, db_path=str(tmp))
    registrar_ejecucion("test-2", 10, 9, 1, db_path=str(tmp))
    con = conectar(str(tmp))
    try:
        n = con.execute("SELECT COUNT(*) FROM etl_control WHERE run_date = ?", (RUN_DATE,)).fetchone()[0]
        _assert(n == 1, "etl_control duplicó la fecha")
    finally:
        con.close()
        shutil.rmtree(tmp.parent, ignore_errors=True)


def run_tests():
    """Ejecuta todos los tests y reporta PASS/FAIL."""
    pruebas = [v for k, v in sorted(globals().items()) if k.startswith("test_") and callable(v)]
    resultados = []
    for fn in pruebas:
        try:
            fn()
            resultados.append((fn.__name__, "PASS", ""))
            log.info("TEST PASS %s", fn.__name__, extra={"step": "S"})
        except Exception as e:
            resultados.append((fn.__name__, "FAIL", str(e)))
            log.error("TEST FAIL %s: %s", fn.__name__, e, extra={"step": "S"})
    df = pd.DataFrame(resultados, columns=["test", "resultado", "detalle"])
    display(df)
    _assert((df["resultado"] == "PASS").all(), "Hay tests fallidos")
    return df

run_tests()

2026-11-03 02:11:02 | INFO     | [S] | TEST PASS test_csv_con_timestamp
2026-11-03 02:11:02 | INFO     | [S] | TEST PASS test_extract_lee_csv
2026-11-03 02:11:02 | INFO     | [E] | EXTRACT inicio (1 tablas)
2026-11-03 02:11:02 | WARNING  | [E] | dim_canal no existe; se regenera Bronze y se persiste
2026-11-03 02:11:02 | INFO     | [E] | Generando datos Bronze (todas las tablas)
2026-11-03 02:11:02 | INFO     | [E] | Bronze dim_fecha → dim_fecha_20990101_000000.csv (365 filas)
2026-11-03 02:11:02 | INFO     | [E] | Bronze dim_categoria → dim_categoria_20990101_000000.csv (5 filas)
2026-11-03 02:11:02 | INFO     | [E] | Bronze dim_producto → dim_producto_20990101_000000.csv (20 filas)
2026-11-03 02:11:02 | INFO     | [E] | Bronze dim_sucursal → dim_sucursal_20990101_000000.csv (8 filas)
2026-11-03 02:11:03 | INFO     | [E] | Bronze dim_cliente → dim_cliente_20990101_000000.csv (40000 filas)
2026-11-03 02:11:03 | INFO     | [E] | Bronze dim_canal → dim_canal_20990101_000000.csv (3 filas)


/tmp/ipykernel_185231/1493486422.py:21: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  "dia_semana":   fechas.dayofweek + 1,
/tmp/ipykernel_185231/1493486422.py:23: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  "es_fin_semana": (fechas.dayofweek >= 5).astype(int),


2026-11-03 02:11:03 | INFO     | [E] | Bronze fact_ventas → fact_ventas_20990101_000000.csv (200000 filas)
2026-11-03 02:11:03 | INFO     | [E] | Bronze fact_inventario → fact_inventario_20990101_000000.csv (12000 filas)
2026-11-03 02:11:03 | INFO     | [E] | Bronze fact_campanas → fact_campanas_20990101_000000.csv (150 filas)
2026-11-03 02:11:03 | INFO     | [E] | Bronze fact_devoluciones → fact_devoluciones_20990101_000000.csv (3000 filas)
2026-11-03 02:11:03 | INFO     | [E] | Bronze completo: 10 tablas
2026-11-03 02:11:03 | INFO     | [E] | EXTRACT dim_canal              3 filas
2026-11-03 02:11:03 | INFO     | [E] | EXTRACT fin
2026-11-03 02:11:03 | INFO     | [S] | TEST PASS test_extract_regenera_si_falta
2026-11-03 02:11:03 | INFO     | [S] | TEST PASS test_generacion_bronze
2026-11-03 02:11:03 | INFO     | [S] | TEST PASS test_idempotencia_diaria
2026-11-03 02:11:03 | INFO     | [S] | TEST PASS test_load_sqlite
2026-11-03 02:11:03 | INFO     | [S] | TEST PASS test_motivos_mapea

,test,resultado,detalle
0,test_csv_con_timestamp,PASS,
1,test_extract_lee_csv,PASS,
2,test_extract_regenera_si_falta,PASS,
3,test_generacion_bronze,PASS,
4,test_idempotencia_diaria,PASS,
5,test_load_sqlite,PASS,
6,test_motivos_mapeados,PASS,
7,test_rechazos_con_motivo,PASS,
8,test_rut_valido_90,PASS,
9,test_transform_rechaza_rut_invalido,PASS,


,test,resultado,detalle
0,test_csv_con_timestamp,PASS,
1,test_extract_lee_csv,PASS,
2,test_extract_regenera_si_falta,PASS,
3,test_generacion_bronze,PASS,
4,test_idempotencia_diaria,PASS,
5,test_load_sqlite,PASS,
6,test_motivos_mapeados,PASS,
7,test_rechazos_con_motivo,PASS,
8,test_rut_valido_90,PASS,
9,test_transform_rechaza_rut_invalido,PASS,


## 8. Ejecución integral

Orquesta los pasos con control de idempotencia. La primera ejecución del día carga;
las siguientes no modifican la base.

In [27]:
def ejecutar_pipeline(ts: str = RUN_TS, run_date: str = RUN_DATE) -> dict:
    """Orquesta EXTRACT → TRANSFORM → LOAD con idempotencia diaria."""
    t0 = time.perf_counter()
    log.info("=" * 60, extra={"step": "S"})
    log.info("INICIO PIPELINE DataRetail 360", extra={"step": "S"})

    if ya_ejecutado_hoy(run_date=run_date):
        log.warning("Pipeline ya ejecutado hoy (%s). Se omite.", run_date, extra={"step": "S"})
        return {"estado": "SIN_CAMBIOS", "run_date": run_date}

    extraidos = extraer(ts)
    dims, rech_dim = validar_dimensiones(extraidos)
    facts, rech_fact = validar_hechos(extraidos, dims)
    guardar_rechazos(rech_dim + rech_fact, ts)

    n_leidas = sum(len(df) for df in extraidos.values())
    n_ok = sum(len(df) for df in list(dims.values()) + list(facts.values()))
    n_rech = n_leidas - n_ok
    conteos = cargar(dims, facts)
    registrar_ejecucion(ts, n_leidas, n_ok, n_rech)

    dur = round(time.perf_counter() - t0, 2)
    log.info("FIN PIPELINE · leídas=%d ok=%d rechazadas=%d · %.2fs", n_leidas, n_ok, n_rech, dur,
             extra={"step": "S"})
    return {"estado": "OK", "leidas": n_leidas, "ok": n_ok, "rechazadas": n_rech,
            "conteos": conteos, "duracion_s": dur}

resultado = ejecutar_pipeline()
print(json.dumps({k: v for k, v in resultado.items() if k != "conteos"}, indent=2, ensure_ascii=False))
print("\nConteos por tabla:")
display(pd.DataFrame([resultado.get("conteos", {})]).T.rename(columns={0: "filas"}))

2026-11-03 02:11:03 | INFO     | [S] | ============================================================
2026-11-03 02:11:03 | INFO     | [S] | INICIO PIPELINE DataRetail 360
2026-11-03 02:11:03 | WARNING  | [S] | Pipeline ya ejecutado hoy (2026-11-03). Se omite.
{
  "estado": "SIN_CAMBIOS",
  "run_date": "2026-11-03"
}

Conteos por tabla:


,filas


In [28]:
# Segunda ejecución el mismo día: debe omitirse (idempotencia)
resultado_2 = ejecutar_pipeline()
print("Segunda corrida:", resultado_2["estado"])

# Consulta analítica de ejemplo sobre el datamart
con = conectar()
try:
    q = """
    SELECT s.region, c.categoria, ROUND(SUM(f.monto_neto), 0) AS venta_neta
    FROM fact_ventas f
    JOIN dim_sucursal s ON s.sucursal_key = f.sucursal_key
    JOIN dim_producto p ON p.producto_key = f.producto_key
    JOIN dim_categoria c ON c.categoria_key = p.categoria_key
    GROUP BY s.region, c.categoria
    ORDER BY venta_neta DESC
    LIMIT 8;
    """
    display(pd.read_sql_query(q, con))
finally:
    con.close()

print("\nArchivos generados:")
for p in sorted(list(DATA_DIR.glob(f"*_{RUN_TS}.csv")) + list(REJECTED_DIR.glob(f"*_{RUN_TS}.csv")) + [LOG_FILE]):
    print(" -", p)

2026-11-03 02:11:03 | INFO     | [S] | ============================================================
2026-11-03 02:11:03 | INFO     | [S] | INICIO PIPELINE DataRetail 360
2026-11-03 02:11:03 | WARNING  | [S] | Pipeline ya ejecutado hoy (2026-11-03). Se omite.
Segunda corrida: SIN_CAMBIOS


,region,categoria,venta_neta
0,Metropolitana,Gaming,2.342330e+10
1,Metropolitana,Electrohogar,1.933459e+10
2,Metropolitana,Tecnología,1.395239e+10
3,Metropolitana,Hogar,7.438249e+09
4,Araucanía,Gaming,5.853389e+09
5,Biobío,Gaming,5.836607e+09
6,Antofagasta,Gaming,5.802151e+09
7,Valparaíso,Gaming,5.801937e+09



Archivos generados:
 - data/dim_canal_20261103_021101.csv
 - data/dim_categoria_20261103_021101.csv
 - data/dim_cliente_20261103_021101.csv
 - data/dim_fecha_20261103_021101.csv
 - data/dim_producto_20261103_021101.csv
 - data/dim_sucursal_20261103_021101.csv
 - data/fact_campanas_20261103_021101.csv
 - data/fact_devoluciones_20261103_021101.csv
 - data/fact_inventario_20261103_021101.csv
 - data/fact_ventas_20261103_021101.csv
 - logs/etl_20261103_021101.log
 - rejected/Rejected_20261103_021101.csv


## 9. Cierre

- **Bronze:** 10 tablas generadas con código y semilla fija; RUT válidos ≥ 90%; CSV con timestamp.
- **EXTRACT:** lee CSV si existen; si no, regenera y persiste.
- **TRANSFORM:** reglas de calidad con `motivo_rechazo` mapeado en código; `Rejected_{timestamp}.csv`.
- **LOAD:** `DataRetail360.db` con DDL exacto de `Tablas.md` (6 dims + 4 facts).
- **Idempotencia:** una corrida por día vía `etl_control`; `replace` evita duplicados.
- **Logging:** `logs/etl_{timestamp}.log` con nivel, paso (E/T/L) y timestamp; se imprime en consola.
- **Tests:** `run_tests()` cubre generación, RUT, CSV, EXTRACT, rechazos, LOAD e idempotencia.

> Para reiniciar el laboratorio desde cero: elimina `DataRetail360.db`, `data/`, `logs/` y `rejected/`, y vuelve a ejecutar todo.